# 03b — full PAD-pretrained MiniFASNetV2 Vanilla
Independent Kaggle run. Attach the same v2 resource bundle to both branches. Val50K alone controls checkpoint and threshold; full CelebA Test and official LCC are evaluated after a saved freeze.


In [ ]:
# CONFIG: attach CelebA, official LCC, MiniFASNet, and the completed v2 resource ZIP as Kaggle datasets.
CELEBA_ROOT="/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof"
LCC_FULL_ROOT="/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD_evaluation" # edit mount if needed
RESOURCE_ROOT="/kaggle/input/<user-mounted-v2-dataset>/minifasnet_full_resources_v2" # edit
MINIFASNET_SOURCE_PATH="/kaggle/input/datasets/takasugi1303/minifasnet/MiniFASNet.py"
MINIFASNET_CHECKPOINT_PATH="/kaggle/input/datasets/takasugi1303/minifasnet/2.7_80x80_MiniFASNetV2.pth"
OUTPUT_ROOT="/kaggle/working/minifasnet_full_pretrained_vanilla_seed100"
RESUME=False; EVALUATION_CONTRACT_VERSION='PAD_DATA_AND_EVALUATION_CONTRACT_v1'
EXPECTED_CONTRACT_SHA256='aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d'
TRAIN_SEED=100; DATA_SEED=42
BATCH_SIZE=128; NUM_WORKERS=4; AMP=True
MAX_EPOCHS=12; EARLY_STOP_PATIENCE=3; EARLIEST_EARLY_STOP_EPOCH=5
BACKBONE_LR=2e-4; CLASSIFIER_LR=1e-3; WEIGHT_DECAY=1e-4; LABEL_SMOOTHING=.1; GRAD_CLIP=5.
CROP_SCALE=2.7; INPUT_SIZE=80; METRIC_TIE_EPS=1e-12; METRIC_VERIFY_EPS=1e-6
CSMR_WARMUP_EPOCHS=1; CSMR_LAMBDA=.4; CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.1; CSMR_GAIN_RANGE=[.65,.90]; CSMR_PRESERVE_DC=True; SELECTION_CHUNK_SIZE=32
TRAIN_MANIFEST='final_train_full'; VAL_MANIFEST='final_val50k'; TEST_MANIFEST='final_test_full'; LCC_MANIFEST='lcc_official_evaluation_full'
TRAIN_EXPECTED_MIN=100000; VAL_EXPECTED=50000; TEST_EXPECTED=67170; LCC_EXPECTED=7580
INIT_NAME='pretrained'; TEST_ROLE='full official CelebA-Spoof Test; held out until Val freeze'
FROZEN_MINIFASNET_SOURCE_SHA256='e498c4ec5e1ddfaba62b941a126c19d65aa564999f3309661fe43ee8bf38acd7'
FROZEN_PRETRAINED_CHECKPOINT_SHA256='a5eb02e1843f19b5386b953cc4c9f011c3f985d0ee2bb9819eea9a142099bec0'
EXPECTED_INITIAL_STATE_SHA256='318dbe2a0fa2c80cc36d4f881aa3967b226255af71a45049529daa5430b93adc'
TUNING_CONFIG_SOURCE='Notebook 03 frozen pretrained recipe';TUNING_CONFIG_SHA256='bfc8652b237d7a8156bf455667bcbc00794baa9b0f0283c4a449406da6caeb0a'
import shutil, sys, subprocess
print('Evaluation contract:',EVALUATION_CONTRACT_VERSION,'SHA256:',EXPECTED_CONTRACT_SHA256)
print('Configured roots:',CELEBA_ROOT,LCC_FULL_ROOT,RESOURCE_ROOT,MINIFASNET_SOURCE_PATH,MINIFASNET_CHECKPOINT_PATH)
print('Python',sys.version.split()[0],'GPU:',subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print('Free disk GB:',round(shutil.disk_usage('/kaggle/working').free/1e9,2))


In [ ]:
# Install ONNX dependencies for export and parity checking
%pip install -q onnx onnxruntime


In [ ]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, time, zipfile, copy, gc
from collections import Counter, OrderedDict
from pathlib import Path, PurePosixPath
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import confusion_matrix, roc_curve
from IPython.display import FileLink, display
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]


def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb


def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)


def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))


def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }


def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores


def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result


def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes


def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full


def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


def strict_json_write(path,payload):
    Path(path).write_text(json.dumps(payload,indent=2,allow_nan=False)+'\n')

def strict_json_load(path):
    return json.loads(Path(path).read_text(),parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))

def ordered_sha(keys):return sha256_text('\n'.join(map(str,keys)))

def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:
        raise ValueError('Val threshold needs finite 3-class labels/scores')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0)
    nr=int(real.sum());na=len(y)-nr;bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)]
    rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]

def resource_tree_sha(root):
    h=hashlib.sha256()
    for p in sorted(Path(root).rglob('*')):
        if p.is_file():h.update(str(p.relative_to(root)).encode()+b'\0'+sha256_file(p).encode()+b'\n')
    return h.hexdigest()
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Official CropImage._get_new_box geometry, adapted from xyxy face coordinates.
    src_h,src_w=image.shape[:2]
    x1,y1,x2,y2=map(float,bbox_xyxy)
    box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:
        raise ValueError('Invalid MiniFASNet image/bbox geometry')
    used_scale=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale))
    new_w,new_h=box_w*used_scale,box_h*used_scale
    center_x,center_y=box_w/2+x1,box_h/2+y1
    left,top=center_x-new_w/2,center_y-new_h/2
    right,bottom=center_x+new_w/2,center_y+new_h/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>src_w-1: left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1: top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:
        raise RuntimeError(f'MiniFASNet crop escaped image: {(lx,ty,rx,by)}')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty MiniFASNet crop')
    return cv2.resize(patch,(size,size)),(lx,ty,rx,by),used_scale


def mini_tensor(patch):
    if patch.shape!=(INPUT_SIZE,INPUT_SIZE,3) or patch.dtype!=np.uint8:
        raise ValueError('Expected 80x80 uint8 BGR patch')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)


def source_module(path):
    spec=importlib.util.spec_from_file_location('local_minifasnet_v2',path)
    if spec is None or spec.loader is None:raise RuntimeError('Cannot import local MiniFASNet.py')
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    return module


def new_model(module):
    return module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),
        drop_p=.2,num_classes=3,img_channel=3)


def state_fingerprint(state):
    h=hashlib.sha256()
    for key in sorted(state):
        value=state[key].detach().cpu().contiguous()
        h.update(key.encode());h.update(str(value.dtype).encode());h.update(str(tuple(value.shape)).encode())
        h.update(value.numpy().tobytes())
    return h.hexdigest()


def load_pretrained_canonical(module,checkpoint_path):
    raw=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Expected official OrderedDict with module. prefix')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if list(stripped['prob.weight'].shape)!=[3,128]:raise RuntimeError('Official prob.weight shape differs from [3,128]')
    if tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official conv6 kernel is not 5x5')
    seed_everything(TRAIN_SEED)
    model=new_model(module)
    model_keys=model.state_dict()
    skipped=['prob.weight']
    shape_mismatch=[k for k,v in stripped.items() if k in model_keys and tuple(v.shape)!=tuple(model_keys[k].shape) and k not in skipped]
    unexpected=[k for k in stripped if k not in model_keys]
    missing_pretrained=[k for k in model_keys if k not in stripped and k not in skipped]
    if shape_mismatch or unexpected or missing_pretrained:raise RuntimeError(f'Unrelated checkpoint mismatch: shape={shape_mismatch[:5]} unexpected={unexpected[:5]} missing={missing_pretrained[:5]}')
    compatible=OrderedDict((k,v) for k,v in stripped.items() if k not in skipped)
    missing,unexpected_load=model.load_state_dict(compatible,strict=False)
    if set(missing)!={'prob.weight'} or unexpected_load:raise RuntimeError(f'Loading audit failed: missing={missing}, unexpected={unexpected_load}')
    model.prob=nn.Linear(128,3,bias=False)  # new task semantics, deterministic seed 100
    if not model.prob.weight.requires_grad or not model.conv1.conv.weight.requires_grad:
        raise RuntimeError('Classifier/backbone must remain trainable')
    audit={'schema':'mini_fasnet_pretrained_loading_audit_v1','checkpoint_file':checkpoint_path.name,
        'checkpoint_sha256':sha256_file(checkpoint_path),'checkpoint_bytes':checkpoint_path.stat().st_size,
        'architecture_source_sha256':sha256_file(MINIFASNET_SOURCE),
        'checkpoint_keys':len(raw),'loaded_pretrained_key_count':len(compatible),
        'missing_keys':list(missing),'unexpected_keys':list(unexpected_load),
        'intentionally_skipped_keys':skipped,'shape_mismatch_keys':shape_mismatch,
        'input_size':[80,80],'crop_scale':CROP_SCALE,'channel_order':'BGR',
        'conv6_kernel':[5,5],'embedding_size':128,'classifier_shape':[3,128],
        'total_params':sum(p.numel() for p in model.parameters()),
        'trainable_params':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model,audit
def require_scrfd_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD':
        raise RuntimeError('Unscorable or non-SCRFD localization: '+key)
    if rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):
        raise RuntimeError('SCRFD bbox provenance mismatch: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Invalid raw SCRFD bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite((x1,y1,x2,y2)).all() or x2<=x1 or y2<=y1:
        raise RuntimeError('Nonfinite/empty SCRFD bbox: '+key)
    return rec

class FinalCelebADataset(Dataset):
    def __init__(self,frame,root,records,train=False):
        self.frame=frame.reset_index(drop=True);self.root=Path(root);self.records=records;self.train=train
        missing=set(self.frame.sample_key.astype(str))-set(records)
        if missing:raise RuntimeError(f'CelebA bbox missing {len(missing)} manifest keys')
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i];key=str(row.sample_key);rec=self.records[key]
        require_scrfd_record(rec,key)
        image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        bbox=bbox_jitter(rec) if self.train else rec['bbox_xyxy']
        patch,_,_=mini_crop_bgr(image,bbox)
        if self.train:patch=apply_e1_mild_augmentation(patch)
        return mini_tensor(patch),int(row.three_class_label),key,int(row.attack_code)

class FinalLCCDataset(Dataset):
    def __init__(self,frame,root,records):
        self.rows=[];self.root=Path(root)
        for row in frame.itertuples(index=False):
            if str(row.status)=='VALID':
                rec=records[str(row.path)]
                require_scrfd_record(rec,str(row.path))
                self.rows.append((str(row.path),int(row.label),rec))
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        key,label,rec=self.rows[i];image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        patch,_,_=mini_crop_bgr(image,rec['bbox_xyxy'])
        return mini_tensor(patch),label,key,'VALID'


In [ ]:
# ONNX preflight before reading 120K resources or starting expensive final training.
import importlib.util, tempfile
import subprocess, sys
missing=[name for name in ('onnx','onnxruntime') if importlib.util.find_spec(name) is None]
if missing:
    print('Installing missing ONNX export dependencies:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
import onnx, onnxruntime as ort
if 'CPUExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('ONNX Runtime CPUExecutionProvider required for parity validation')
try:
    with tempfile.TemporaryDirectory() as temp_dir:
        smoke_path=Path(temp_dir)/'export_smoke.onnx'
        torch.onnx.export(torch.nn.Identity().eval(),torch.zeros(1,3),str(smoke_path),
            opset_version=17,input_names=['input'],output_names=['logits'],dynamo=False)
        onnx.checker.check_model(onnx.load(str(smoke_path)))
        smoke_session=ort.InferenceSession(str(smoke_path),providers=['CPUExecutionProvider'])
        if smoke_session.run(['logits'],{'input':np.zeros((1,3),dtype=np.float32)})[0].shape!=(1,3):
            raise RuntimeError('ONNX Runtime smoke output shape differs')
except Exception as exc:
    raise RuntimeError('Torch ONNX legacy exporter/ORT preflight failed before training') from exc
print('ONNX preflight PASS:',onnx.__version__,ort.__version__,'Torch:',torch.__version__,
      'providers:',ort.get_available_providers(),'dynamo=False')


In [ ]:
root=Path(RESOURCE_ROOT);output=Path(OUTPUT_ROOT);output.mkdir(parents=True,exist_ok=True)
celeba_root=Path(CELEBA_ROOT);lcc_root=Path(LCC_FULL_ROOT);MINIFASNET_SOURCE=Path(MINIFASNET_SOURCE_PATH)
if (lcc_root/'LCC_FASD_evaluation').is_dir():lcc_root=lcc_root/'LCC_FASD_evaluation'
print('Resolved LCC evaluation root:',lcc_root)
required=[root/'config.json',root/'provenance.json',root/'manifest_fingerprints.json',
    root/(EVALUATION_CONTRACT_VERSION+'.md'),root/'bbox/celeba_test_localization_audit.json',
    root/'manifests/final_train_full.csv',root/'manifests/final_val50k.csv',root/'manifests/final_test_full.csv',
    root/'manifests/lcc_official_evaluation_full.csv',root/'bbox/celeba_bbox_cache_full.json',
    root/'bbox/lcc_official_evaluation_bbox_cache.json',celeba_root,lcc_root,MINIFASNET_SOURCE,Path(MINIFASNET_CHECKPOINT_PATH)]
for path in required:
    if not path.exists():raise FileNotFoundError(f'Kaggle mount missing: {path}. Edit CONFIG and attach required dataset.')
if sha256_file(MINIFASNET_SOURCE)!=FROZEN_MINIFASNET_SOURCE_SHA256:raise RuntimeError('MiniFASNet source SHA mismatch')
if sha256_file(MINIFASNET_CHECKPOINT_PATH)!=FROZEN_PRETRAINED_CHECKPOINT_SHA256:raise RuntimeError('Official pretrained checkpoint SHA mismatch')
resource_config=strict_json_load(root/'config.json');resource_provenance=strict_json_load(root/'provenance.json')
contract_path=root/(EVALUATION_CONTRACT_VERSION+'.md')
contract_sha=sha256_file(contract_path)
if contract_sha!=EXPECTED_CONTRACT_SHA256:raise RuntimeError('Mounted evaluation contract SHA mismatch')
if resource_config.get('evaluation_contract_version')!=EVALUATION_CONTRACT_VERSION or resource_provenance.get('evaluation_contract_version')!=EVALUATION_CONTRACT_VERSION:
    raise RuntimeError('Resource contract version mismatch')
if resource_config.get('evaluation_contract_sha256')!=contract_sha or resource_provenance.get('evaluation_contract_sha256')!=contract_sha:
    raise RuntimeError('Resource contract SHA mismatch')
if resource_config['celeba_bbox_policy'].get('fallback')!='none' or resource_config['lcc_policy'].get('fallback')!='none':
    raise RuntimeError('Resource still permits GT fallback')
if resource_provenance.get('scrfd_execution_provider')!='CUDAExecutionProvider':
    raise RuntimeError('Resource localization must be fresh CUDA SCRFD')
if (resource_config['celeba_bbox_policy'].get('provider')!=['CUDAExecutionProvider'] or
    resource_config['lcc_policy'].get('provider')!=['CUDAExecutionProvider']):
    raise RuntimeError('Resource cache policies must be CUDA-only')
print('Verified evaluation contract:',EVALUATION_CONTRACT_VERSION,'SHA256:',contract_sha)

train_meta_path=celeba_root/'metas/intra_test/train_label.json';test_meta_path=celeba_root/'metas/intra_test/test_label.json'
if sha256_file(train_meta_path)!=resource_provenance['source_hashes']['train_label_json'] or sha256_file(test_meta_path)!=resource_provenance['source_hashes']['test_label_json']:
    raise RuntimeError('Mounted CelebA metadata differs from resource preparation')
if resource_config['input']!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}:raise RuntimeError('Resource preprocessing contract differs')
manifest_fp=strict_json_load(root/'manifest_fingerprints.json');frames={}
for name in (TRAIN_MANIFEST,VAL_MANIFEST,TEST_MANIFEST):
    path=root/'manifests'/(name+'.csv');fp=manifest_fp[name]
    if sha256_file(path)!=fp['file_sha256']:raise RuntimeError('Manifest SHA mismatch: '+name)
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=fp['sample_count'] or ordered_sha(frame.sample_key.astype(str))!=fp['ordered_sample_key_sha256']:raise RuntimeError('Manifest fingerprint mismatch: '+name)
    if set(frame.three_class_label.astype(int))!={0,1,2}:raise RuntimeError('Three-class manifest missing class: '+name)
    frames[name]=frame
train_frame,val_frame,test=frames[TRAIN_MANIFEST],frames[VAL_MANIFEST],frames[TEST_MANIFEST]
if not(len(train_frame)>TRAIN_EXPECTED_MIN and len(val_frame)==VAL_EXPECTED and len(test)==TEST_EXPECTED):raise RuntimeError('Full CelebA manifest population mismatch')
if set(train_frame.sample_key)&set(val_frame.sample_key) or (set(train_frame.sample_key)|set(val_frame.sample_key))&set(test.sample_key):raise RuntimeError('Train/Val/Test key overlap')
if not set(train_frame.subject_id.astype(str)).isdisjoint(set(val_frame.subject_id.astype(str))|set(test.subject_id.astype(str))):raise RuntimeError('Train subject leakage')
lcc_path=root/'manifests'/(LCC_MANIFEST+'.csv');lcc_frame=pd.read_csv(lcc_path,keep_default_na=False)
if sha256_file(lcc_path)!=manifest_fp[LCC_MANIFEST]['file_sha256'] or len(lcc_frame)!=LCC_EXPECTED or (lcc_frame.label==0).sum()!=314 or (lcc_frame.label==1).sum()!=7266:raise RuntimeError('Official LCC manifest mismatch')
bbox_path=root/'bbox/celeba_bbox_cache_full.json';lcc_bbox_path=root/'bbox/lcc_official_evaluation_bbox_cache.json'
if sha256_file(bbox_path)!=resource_provenance['celeba_bbox_sha256'] or sha256_file(lcc_bbox_path)!=resource_provenance['lcc_bbox_sha256']:raise RuntimeError('Bbox cache SHA mismatch')
celeba_payload=strict_json_load(bbox_path);lcc_payload=strict_json_load(lcc_bbox_path)
for payload in (celeba_payload,lcc_payload):
    header=payload.get('header',{})
    if (header.get('provider')!='CUDAExecutionProvider' or
        header.get('detector_sha256')!=resource_provenance.get('scrfd_model_sha256') or
        header.get('contract_sha256')!=contract_sha):
        raise RuntimeError('BBox cache lacks homogeneous CUDA SCRFD provenance')
bbox_cache=celeba_payload['records'];lcc_cache=lcc_payload['records']
for key,rec in bbox_cache.items():
    if rec.get('usable_for_pad'):require_scrfd_record(rec,key)
for key,rec in lcc_cache.items():
    if rec.get('usable_for_pad'):require_scrfd_record(rec,key)
required_celeba=set(train_frame.sample_key.astype(str))|set(val_frame.sample_key.astype(str))|set(test.sample_key.astype(str))
if not required_celeba<=set(bbox_cache) or not set(lcc_frame.path.astype(str))<=set(lcc_cache):raise RuntimeError('Bbox cache does not cover manifests')
if any('crop_factor' in r for r in bbox_cache.values()) or any('crop_factor' in r for r in lcc_cache.values()):raise RuntimeError('BBox cache bakes crop factor')
if set(bbox_cache)!=set(json.loads(train_meta_path.read_text()))|set(json.loads(test_meta_path.read_text())):
    raise RuntimeError('CelebA full bbox cache does not cover official metadata')
test_audit=strict_json_load(root/'bbox/celeba_test_localization_audit.json')
if test_audit['total_candidates']!=67170:raise RuntimeError('Test candidate audit population mismatch')
for frame in (train_frame,val_frame,test):
    for row in frame.itertuples(index=False):
        key=str(row.sample_key);rec=bbox_cache[key]
        if str(row.localization_status)!=rec['status'] or bool(row.usable_for_pad)!=bool(rec['usable_for_pad']) or str(row.bbox_origin)!=rec['bbox_origin']:
            raise RuntimeError('CelebA manifest/cache status mismatch: '+key)
        if rec['usable_for_pad']:require_scrfd_record(rec,key)
        elif rec.get('bbox_xyxy') is not None or rec.get('bbox_origin')!='NONE':
            raise RuntimeError('Unscorable CelebA candidate holds a bbox: '+key)
for row in lcc_frame.itertuples(index=False):
    key=str(row.path);rec=lcc_cache[key]
    if str(row.status)!=rec['status'] or bool(row.usable_for_pad)!=bool(rec['usable_for_pad']) or str(row.bbox_origin)!=rec['bbox_origin']:
        raise RuntimeError('LCC manifest/cache status mismatch: '+key)
    if rec['usable_for_pad']:require_scrfd_record(rec,key)
    elif rec.get('bbox_xyxy') is not None or rec.get('bbox_origin')!='NONE':
        raise RuntimeError('Unscorable LCC candidate holds a bbox: '+key)

for key in train_frame.sample_key.astype(str):
    rec=require_scrfd_record(bbox_cache[key],key)
    x1,y1,x2,y2=map(float,rec['bbox_xyxy'])
    if min(x2-x1,y2-y1)<48:raise RuntimeError('Train bbox too small: '+key)
for key in val_frame.sample_key.astype(str):require_scrfd_record(bbox_cache[key],key)
if len(val_frame)!=50000 or not val_frame.usable_for_pad.astype(bool).all():raise RuntimeError('Val50K is not entirely SCRFD-scorable')
test_frame=test[test.usable_for_pad.astype(bool)].reset_index(drop=True)
if len(test_frame)!=test_audit['scored_n'] or len(test_frame)!=test_audit['valid_scrfd']:
    raise RuntimeError('Test scored count differs from localization audit')
lcc_scored=lcc_frame[lcc_frame.usable_for_pad.astype(bool)]
if len(lcc_scored)!=int((lcc_frame.status=='VALID').sum()):raise RuntimeError('LCC scored status mismatch')
for frame in (train_frame,val_frame,test_frame):
    if not all((celeba_root/k).is_file() for k in frame.sample_key.astype(str)):
        raise FileNotFoundError('Scorable CelebA image missing')
if not all((lcc_root/p).is_file() for p in lcc_scored.path.astype(str)):
    raise FileNotFoundError('Scorable LCC image missing')
resource_hashes={'evaluation_contract_version':EVALUATION_CONTRACT_VERSION,
 'evaluation_contract_sha256':contract_sha,'train_manifest_sha256':sha256_file(root/'manifests'/(TRAIN_MANIFEST+'.csv')),
 'val_manifest_sha256':sha256_file(root/'manifests'/(VAL_MANIFEST+'.csv')),
 'test_manifest_sha256':sha256_file(root/'manifests'/(TEST_MANIFEST+'.csv')),
 'lcc_manifest_sha256':sha256_file(lcc_path),'celeba_bbox_sha256':sha256_file(bbox_path),
 'lcc_bbox_sha256':sha256_file(lcc_bbox_path),'minifasnet_source_sha256':sha256_file(MINIFASNET_SOURCE),
 'official_checkpoint_sha256':sha256_file(MINIFASNET_CHECKPOINT_PATH),
 'manifest_fingerprints':{n:manifest_fp[n] for n in (TRAIN_MANIFEST,VAL_MANIFEST,TEST_MANIFEST,LCC_MANIFEST)}}
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');use_amp=bool(AMP and device.type=='cuda')
seed_everything(TRAIN_SEED);module=source_module(MINIFASNET_SOURCE)
initial_model,loading_audit=load_pretrained_canonical(module,Path(MINIFASNET_CHECKPOINT_PATH))
canonical_state={k:v.detach().cpu().clone() for k,v in initial_model.state_dict().items()};initial_sha=state_fingerprint(canonical_state)
if initial_sha!=EXPECTED_INITIAL_STATE_SHA256:raise RuntimeError(f'Canonical initialization SHA mismatch: {initial_sha}')
params=sum(p.numel() for p in initial_model.parameters())
if not 300000<=params<=600000:raise RuntimeError('MiniFASNetV2 parameter count differs')
with torch.no_grad():
    initial_model.eval();assert tuple(initial_model(torch.zeros(2,3,80,80)).shape)==(2,3)
del initial_model
strict_json_write(output/'resource_fingerprints.json',resource_hashes)
strict_json_write(output/'initialization_fingerprint.json',{'sha256':initial_sha,'params':params,'trainable_params':params,'loading_audit':loading_audit})
print('Packages:',{'torch':torch.__version__,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__})
print('Data:',{'train':len(train_frame),'val':len(val_frame),'test_candidates':len(test),'test_scored':len(test_frame),'lcc_candidates':len(lcc_frame),'lcc_scored':len(lcc_scored)},'initial SHA:',initial_sha,'device:',device)


In [ ]:
BRANCH='vanilla';CSMR_ENABLED=False


In [ ]:
def branch_config(name,csmr_enabled):
    return {'schema':'minifasnet_full_pretrained_v2','evaluation_contract_version':EVALUATION_CONTRACT_VERSION,
        'evaluation_contract_sha256':contract_sha,'localization_policy':'SCRFD-only; no GT fallback','branch':name,'initialization':INIT_NAME,'train_seed':TRAIN_SEED,
        'data_seed':DATA_SEED,'resource_hashes':resource_hashes,'initial_state_sha256':initial_sha,
        'train_manifest':TRAIN_MANIFEST,'val_manifest':VAL_MANIFEST,'test_manifest':TEST_MANIFEST,
        'test_role':TEST_ROLE,'checkpoint_selection':'lowest Val ACER; tie highest Val AUC',
        'metric_tie_eps':METRIC_TIE_EPS,
        'hyperparameter_selection':{'source':TUNING_CONFIG_SOURCE,'frozen_hparams_sha256':TUNING_CONFIG_SHA256,'policy':'no Full-scale retuning'},
        'input':[3,80,80],'crop_scale':CROP_SCALE,'channels':'BGR','range':'[0,1]',
        'max_epochs':MAX_EPOCHS,'patience':EARLY_STOP_PATIENCE,'earliest_stop':EARLIEST_EARLY_STOP_EPOCH,
        'batch_size':BATCH_SIZE,'amp':use_amp,'optimizer':'AdamW','backbone_lr':BACKBONE_LR,
        'classifier_lr':CLASSIFIER_LR,'weight_decay':WEIGHT_DECAY,'label_smoothing':LABEL_SMOOTHING,
        'grad_clip':GRAD_CLIP,'augmentation':'E1 bbox jitter .2; flip .5; brightness/contrast .3',
        'csmr':{'warmup':CSMR_WARMUP_EPOCHS,'lambda':CSMR_LAMBDA,'centers':CSMR_BAND_CENTERS,
                'sigma':CSMR_BAND_SIGMA,'gain':CSMR_GAIN_RANGE,'preserve_dc':CSMR_PRESERVE_DC}
                if csmr_enabled else None}

def loader_for(frame,train=False,epoch=0):
    dataset=FinalCelebADataset(frame,celeba_root,bbox_cache,train=train)
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(worker_id):
        value=TRAIN_SEED+epoch*1000+worker_id
        random.seed(value);np.random.seed(value);torch.manual_seed(value)
    return DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=train,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=device.type=='cuda',generator=gen,worker_init_fn=worker_init)

def run_branch(name,csmr_enabled):
    folder=output
    conf=branch_config(name,csmr_enabled);conf_hash=sha256_text(json.dumps(conf,sort_keys=True))
    config_path=folder/'config.json';last_path=folder/'last_state.pth';best_path=folder/'best.pth';meta_path=folder/'meta.json'
    if config_path.exists():
        if strict_json_load(config_path)!=conf:raise RuntimeError(f'{name} config or checkpoint policy differs; restart paired run in a clean output directory')
    else:strict_json_write(config_path,conf)
    seed_everything(TRAIN_SEED)
    model=new_model(module);model.load_state_dict(canonical_state,strict=True)
    if state_fingerprint(model.state_dict())!=initial_sha:raise RuntimeError('Paired initial state drift')
    model=model.to(device)
    if INIT_NAME=='pretrained':
        backbone=[p for n,p in model.named_parameters() if not n.startswith('prob.')]
        groups=[{'params':backbone,'lr':BACKBONE_LR},{'params':list(model.prob.parameters()),'lr':CLASSIFIER_LR}]
    else:groups=[{'params':model.parameters(),'lr':BACKBONE_LR}]
    optimizer=torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)
    scheduler=None
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    history=[];diagnostics=[];class_rows=[];best_epoch=0;best_acer=float('inf');best_auc=-float('inf');start_epoch=1;resume_stopped=False
    if meta_path.exists():
        meta=strict_json_load(meta_path)
        if meta['config_hash']!=conf_hash or not best_path.is_file():raise RuntimeError('Completed branch provenance mismatch')
        checkpoint=torch.load(best_path,map_location='cpu',weights_only=True)
        if sha256_file(best_path)!=meta['checkpoint_sha256']:raise RuntimeError('Completed best SHA mismatch')
        if 'best_val_acer' not in meta or 'best_val_auc' not in meta:
            raise RuntimeError('Completed metadata predates Val AUC tie-break; restart this branch')
        if (checkpoint.get('epoch')!=meta['selected_epoch'] or
            abs(checkpoint.get('best_val_acer',float('inf'))-meta['best_val_acer'])>METRIC_VERIFY_EPS or
            abs(checkpoint.get('best_val_auc',-float('inf'))-meta['best_val_auc'])>METRIC_VERIFY_EPS):
            raise RuntimeError('Completed best checkpoint/metadata selection mismatch')
        model.load_state_dict(checkpoint['state_dict'])
        logits,y,keys,_=eval_spatial(model.to(device),loader_for(val_frame),device,use_amp)
        assert_ordered_keys(keys,val_frame.sample_key.astype(str).tolist(),'Completed frozen Val')
        threshold=threshold_from_val(y,binary_score(torch.from_numpy(logits)).numpy())
        verified,_=metrics_celeba(y,logits,threshold)
        if (abs(verified['acer']-meta['best_val_acer'])>METRIC_VERIFY_EPS or
            abs(verified['auc']-meta['best_val_auc'])>METRIC_VERIFY_EPS):
            raise RuntimeError('Completed best checkpoint fails frozen Val metric verification')
        model.cpu()
        return {'name':name,'model':model,'folder':folder,'val':verified,
                'locked':meta['locked_val_threshold'],'epoch':meta['selected_epoch'],
                'best_acer':meta['best_val_acer'],'best_auc':meta['best_val_auc'],
                'sha':meta['checkpoint_sha256'],'initial_sha':initial_sha}
    if last_path.exists():
        if not RESUME:raise RuntimeError(f'{name} incomplete state exists; set RESUME=True or use clean output dir')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state['config_hash']!=conf_hash or state['branch']!=name or state['initial_state_sha256']!=initial_sha or state['resource_hashes']!=resource_hashes:
            raise RuntimeError('Resume provenance mismatch')
        model.load_state_dict(state['model_state']);optimizer.load_state_dict(state['optimizer_state'])
        scaler.load_state_dict(state['scaler_state'])
        history=state['history'];diagnostics=state['diagnostics'];class_rows=state['class_diagnostics']
        if 'best_auc' not in state:raise RuntimeError('Old resume state lacks best_auc; restart paired run')
        best_epoch=state['best_epoch'];best_acer=state['best_acer'];best_auc=state['best_auc'];start_epoch=state['epoch']+1
        resume_stopped=state.get('stop_reached',False)
        random.setstate(state['rng_python']);np.random.set_state(state['rng_numpy']);torch.set_rng_state(state['rng_torch'])
        if device.type=='cuda' and state['rng_cuda'] is not None:torch.cuda.set_rng_state_all(state['rng_cuda'])
        if not best_path.is_file() or str(best_path)!=state['best_checkpoint_path'] or sha256_file(best_path)!=state['best_checkpoint_sha256']:raise RuntimeError('Resume best checkpoint path/SHA mismatch')
    val_loader=loader_for(val_frame)
    try:
        for epoch in ([] if resume_stopped else range(start_epoch,MAX_EPOCHS+1)):
            started=time.monotonic();model.train();train_loader=loader_for(train_frame,train=True,epoch=epoch)
            losses=[];ce_clean=[];ce_worst=[];cls_losses=[];margin_losses=[]
            band_count=torch.zeros(3,dtype=torch.int64);class_band=torch.zeros((3,3),dtype=torch.int64)
            class_n=torch.zeros(3,dtype=torch.int64);class_harm=torch.zeros(3,dtype=torch.int64)
            class_drop=torch.zeros(3,dtype=torch.float64)
            n_csmr=0;harm_n=0;harm_drop=clean_sum=worst_sum=gain_sum=clip_sum=0.
            print(f'{name} epoch {epoch}/{MAX_EPOCHS}',flush=True)
            for bi,(x,y,_,_) in enumerate(train_loader,1):
                x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits=model(x);cls_loss=criterion(logits,y);margin_loss=cls_loss.detach()*0;loss=cls_loss
                ce_clean.append(float(cls_loss.detach().cpu()))
                if not torch.isfinite(loss):raise RuntimeError('Nonfinite train loss')
                scaler.scale(loss).backward();scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(optimizer);scaler.update()
                losses.append(float(loss.detach().cpu()));cls_losses.append(float(cls_loss.detach().cpu()))
                margin_losses.append(float(margin_loss.detach().cpu()))
                if bi==1 or bi%max(1,len(train_loader)//10)==0:print(f'  batch {bi}/{len(train_loader)} loss {np.mean(losses):.4f}',flush=True)
            val_logits,val_y,val_keys,_=eval_spatial(model,val_loader,device,use_amp)
            assert_ordered_keys(val_keys,val_frame.sample_key.astype(str).tolist(),'Val')
            val_score=binary_score(torch.from_numpy(val_logits)).numpy();threshold=threshold_from_val(val_y,val_score)
            val_metrics,_=metrics_celeba(val_y,val_logits,threshold)
            history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'val_acer':val_metrics['acer'],
                'val_auc':val_metrics['auc'],'val_threshold':threshold,'seconds':time.monotonic()-started})
            val_acer=float(val_metrics['acer']);val_auc=float(val_metrics['auc'])
            better=(val_acer<best_acer-METRIC_TIE_EPS or
                (abs(val_acer-best_acer)<=METRIC_TIE_EPS and val_auc>best_auc+METRIC_TIE_EPS))
            if better:
                best_acer=val_acer;best_auc=val_auc;best_epoch=epoch
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                torch.save({'state_dict':best_state,'epoch':epoch,
                    'best_val_acer':best_acer,'best_val_auc':best_auc},best_path.with_suffix('.pth.tmp'))
                os.replace(best_path.with_suffix('.pth.tmp'),best_path)
            state={'branch':name,'config_hash':conf_hash,'initial_state_sha256':initial_sha,'epoch':epoch,
                'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'resource_hashes':resource_hashes,'best_checkpoint_sha256':sha256_file(best_path),'best_checkpoint_path':str(best_path),'early_stop_counter':epoch-best_epoch,
                'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict() if scheduler else None,
                'scaler_state':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'history':history,'diagnostics':diagnostics,'class_diagnostics':class_rows,
                'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),'rng_torch':torch.get_rng_state(),
                'rng_cuda':torch.cuda.get_rng_state_all() if device.type=='cuda' else None,
                'stop_reached':bool(epoch==MAX_EPOCHS or (epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE))}
            torch.save(state,last_path.with_suffix('.pth.tmp'))
            os.replace(last_path.with_suffix('.pth.tmp'),last_path)
            print(f'  Val ACER {val_metrics["acer"]:.6f}, AUC {val_metrics["auc"]:.6f}; best epoch {best_epoch}, best ACER {best_acer:.6f}, best AUC {best_auc:.6f}',flush=True)
            if epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE:break
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('OOM: restart both branches from clean output with BATCH_SIZE=64') from exc
    if not best_path.is_file():raise RuntimeError('No Val-selected best checkpoint')
    checkpoint=torch.load(best_path,map_location='cpu',weights_only=True)
    if (checkpoint.get('epoch')!=best_epoch or
        abs(checkpoint.get('best_val_acer',float('inf'))-best_acer)>METRIC_VERIFY_EPS or
        abs(checkpoint.get('best_val_auc',-float('inf'))-best_auc)>METRIC_VERIFY_EPS):
        raise RuntimeError('Saved best checkpoint selection metadata mismatch')
    model.load_state_dict(checkpoint['state_dict'])
    logits,y,keys,codes=eval_spatial(model,val_loader,device,use_amp)
    assert_ordered_keys(keys,val_frame.sample_key.astype(str).tolist(),'Frozen Val')
    locked=threshold_from_val(y,binary_score(torch.from_numpy(logits)).numpy())
    metrics,_=metrics_celeba(y,logits,locked)
    if (abs(metrics['acer']-best_acer)>METRIC_VERIFY_EPS or
        abs(metrics['auc']-best_auc)>METRIC_VERIFY_EPS):
        raise RuntimeError('Reloaded best checkpoint fails Val ACER/AUC verification')
    print(f'FROZEN {name}: best_epoch={best_epoch}, best_val_acer={best_acer:.8f}, best_val_auc={best_auc:.8f}')
    make_prediction_frame(keys,y,codes,logits,locked).to_csv(folder/'val_predictions.csv',index=False)
    pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
    if csmr_enabled:
        pd.DataFrame(diagnostics).to_csv(folder/'csmr_diagnostics.csv',index=False)
        pd.DataFrame(class_rows).to_csv(folder/'class_diagnostics.csv',index=False)
    meta={'branch':name,'config_hash':conf_hash,'selected_epoch':best_epoch,'checkpoint_sha256':sha256_file(best_path),
        'initial_state_sha256':initial_sha,'locked_val_threshold':locked,'threshold_source':'CelebA-Spoof Val50K only',
        'best_val_acer':best_acer,'best_val_auc':best_auc,'test_role':TEST_ROLE,'val':metrics}
    strict_json_write(meta_path,meta)
    model.cpu();
    if device.type=='cuda':torch.cuda.empty_cache()
    return {'name':name,'model':model,'folder':folder,'val':metrics,'locked':locked,'epoch':best_epoch,
        'sha':meta['checkpoint_sha256'],'initial_sha':initial_sha,
        'best_acer':best_acer,'best_auc':best_auc}



In [ ]:
result=run_branch(BRANCH,CSMR_ENABLED)
freeze_path=output/'val_freeze.json'
if not (output/'meta.json').is_file() or not (output/'best.pth').is_file():raise RuntimeError('Val freeze incomplete')
freeze={'branch':BRANCH,'checkpoint_sha256':result['sha'],'initial_state_sha256':initial_sha,
        'locked_val_threshold':result['locked'],'threshold_source':'CelebA-Spoof Val50K only',
        'resource_hashes':resource_hashes,'best_epoch':result['epoch'],'best_val_acer':result['best_acer'],'best_val_auc':result['best_auc']}
if freeze_path.is_file() and strict_json_load(freeze_path)!=freeze:raise RuntimeError('Existing Val freeze differs')
strict_json_write(freeze_path,freeze)
print('VAL FROZEN:',freeze)


In [ ]:
# Held-out datasets are accessed only after saved Val freeze and checkpoint SHA agree.
freeze=strict_json_load(output/'val_freeze.json')
if freeze['checkpoint_sha256']!=sha256_file(output/'best.pth') or freeze['resource_hashes']!=resource_hashes:raise RuntimeError('Val freeze provenance mismatch')
model=new_model(module).to(device)
model.load_state_dict(torch.load(output/'best.pth',map_location='cpu',weights_only=True)['state_dict']);model.eval()
locked=float(freeze['locked_val_threshold'])
from sklearn.metrics import confusion_matrix, roc_curve

def roc_stats(y_real,scores):
    fpr,tpr,thresholds=roc_curve(np.asarray(y_real,dtype=int),np.asarray(scores,dtype=float))
    fnr=1-tpr;j=int(np.argmin(np.abs(fnr-fpr)))
    # Linear interpolation at crossing where available.
    delta=fnr-fpr;cross=np.flatnonzero(delta[:-1]*delta[1:]<=0)
    if len(cross):
        j=int(cross[0]);weight=abs(delta[j])/(abs(delta[j])+abs(delta[j+1])) if abs(delta[j])+abs(delta[j+1]) else 0
        eer=float(fpr[j]+weight*(fpr[j+1]-fpr[j]))
    else:eer=float((fnr[j]+fpr[j])/2)
    return {'eer':eer,'tpr_at_fpr_1pct':float(np.max(tpr[fpr<=.01]))}

def save_roc(path,labels,scores,title):
    fpr,tpr,_=roc_curve(np.asarray(labels,dtype=int),np.asarray(scores,dtype=float))
    fig,ax=plt.subplots(figsize=(5,5));ax.plot(fpr,tpr);ax.plot([0,1],[0,1],'--',color='gray')
    ax.set(xlabel='FPR',ylabel='TPR',title=title);fig.tight_layout();fig.savefig(path,dpi=140);plt.close(fig)

metrics_all={}
for split,frame,candidates in (('val',val_frame,val_frame),('test',test_frame,test)):
    logits,y,keys,codes=eval_spatial(model,loader_for(frame),device,use_amp)
    assert_ordered_keys(keys,frame.sample_key.astype(str).tolist(),split)
    metrics,scores=metrics_celeba(y,logits,locked)
    metrics.update(roc_stats(y==0,scores))
    metrics['candidate_n']=len(candidates)
    metrics['scored_n']=len(frame)
    metrics['detector_coverage']=len(frame)/len(candidates)
    metrics['n']=len(frame)
    cm=confusion_matrix(y,np.argmax(logits,axis=1),labels=[0,1,2])
    strict_json_write(output/(split+'_confusion.json'),{'labels':['Real','Physical','Digital'],
        'matrix':cm.tolist(),'scored_n':len(frame)})
    scored=make_prediction_frame(keys,y,codes,logits,locked)
    if split=='test':
        candidates_table=candidates[['sample_key','three_class_label','attack_code','localization_status',
                                     'usable_for_pad','bbox_origin']].rename(columns={
            'sample_key':'key','three_class_label':'true_class'})
        pred=candidates_table.merge(scored.drop(columns=['true_class','attack_code']),
                                    on='key',how='left',validate='one_to_one',sort=False)
        pred['scored']=pred['usable_for_pad'].astype(bool)
        if pred.loc[pred.scored,'pad_logit_score'].isna().any() or pred.loc[~pred.scored,'pad_logit_score'].notna().any():
            raise RuntimeError('Test candidate/scored prediction join differs')
        pred.to_csv(output/'test_predictions.csv',index=False)
    else:
        scored.to_csv(output/'val_predictions.csv',index=False)
    strict_json_write(output/(split+'_metrics.json'),metrics)
    save_roc(output/('roc_'+split+'.png'),y==0,scores,split+' ROC')
    metrics_all[split]=metrics
assert metrics_all['test']['candidate_n']==67170
assert metrics_all['test']['scored_n']==test_audit['scored_n']
assert abs(metrics_all['test']['detector_coverage']-test_audit['detector_coverage'])<1e-12
lcc_dataset=FinalLCCDataset(lcc_frame,lcc_root,lcc_cache)
lcc_loader=DataLoader(lcc_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
logits,y,keys,full=eval_lcc_spatial(model,lcc_loader,device,use_amp,lcc_frame,locked)
score=binary_score(torch.from_numpy(logits)).numpy()
lcc_metrics=metrics_binary(y,score,locked,coverage=len(y)/len(lcc_frame),candidate_n=len(lcc_frame))
lcc_metrics.update(roc_stats(y==0,score));lcc_metrics['candidate_n']=7580;lcc_metrics['scored_n']=len(y)
assert len(lcc_frame)==7580 and len(y)==len(lcc_scored)
full['dataset_role']='LCC-FASD official evaluation split; a smaller resource was previously used in development.'
full['true_label']=full['label'];full['localization_status']=full['status'];full['scored']=full['status'].eq('VALID')
if full.loc[full.scored,'pad_logit_score'].isna().any() or full.loc[~full.scored,'pad_logit_score'].notna().any():
    raise RuntimeError('LCC candidate/scored prediction join differs')
full.to_csv(output/'lcc_predictions.csv',index=False)
strict_json_write(output/'lcc_metrics.json',lcc_metrics);save_roc(output/'roc_lcc.png',y==0,score,'LCC official evaluation ROC')
metrics_all['lcc']=lcc_metrics
strict_json_write(output/'metrics_summary.json',metrics_all)
pd.DataFrame([{'split':name,**values} for name,values in metrics_all.items()]).to_csv(output/'metrics_summary.csv',index=False)
print('Final metrics:',metrics_all)


In [ ]:
# ONNX export only after both Val-frozen checkpoints and held-out evaluation.
import onnx, onnxruntime as ort
if not (output/'val_freeze.json').is_file():raise RuntimeError('Val freeze missing')

def latency(session,device_name='CPU'):
    rng=np.random.default_rng(100);x=rng.random((1,3,80,80),dtype=np.float32)
    for _ in range(20):session.run(['logits'],{'input':x})
    times=[]
    for _ in range(100):
        t=time.perf_counter();session.run(['logits'],{'input':x});times.append((time.perf_counter()-t)*1000)
    return {'provider':device_name,'batch_size':1,'warmup_runs':20,'timed_runs':100,
        'mean_ms':float(np.mean(times)),'median_ms':float(np.median(times)),
        'p95_ms':float(np.percentile(times,95)),'fps':float(1000/np.mean(times))}

# Fixed Val indices. Same tensors feed eval-mode PyTorch and ORT.
val_dataset=FinalCelebADataset(val_frame,celeba_root,bbox_cache,train=False)
indices=np.linspace(0,len(val_dataset)-1,num=8,dtype=int).tolist()
validation_batch=torch.stack([val_dataset[i][0] for i in indices]).contiguous()
efficiency=[]
for result in (result,):
    folder=result['folder'];path=folder/'best.onnx';pending=folder/'best.onnx.pending'
    model=new_model(module);best=torch.load(folder/'best.pth',map_location='cpu',weights_only=True)
    model.load_state_dict(best['state_dict'],strict=True);model.eval()
    with torch.no_grad():expected=model(validation_batch).cpu().numpy()
    torch.onnx.export(model,torch.zeros(1,3,80,80),str(pending),input_names=['input'],output_names=['logits'],
        opset_version=17,dynamic_axes={'input':{0:'batch'},'logits':{0:'batch'}},do_constant_folding=True,dynamo=False)
    graph=onnx.load(str(pending));onnx.checker.check_model(graph)
    if max(x.version for x in graph.opset_import if x.domain=='')!=17:raise RuntimeError('ONNX opset differs')
    if len(graph.graph.input)!=1 or graph.graph.input[0].name!='input' or len(graph.graph.output)!=1 or graph.graph.output[0].name!='logits':
        raise RuntimeError('ONNX IO contract differs')
    in_shape=graph.graph.input[0].type.tensor_type.shape.dim;out_shape=graph.graph.output[0].type.tensor_type.shape.dim
    if [d.dim_value for d in in_shape[1:]]!=[3,80,80] or [d.dim_value for d in out_shape[1:]]!=[3] or in_shape[0].dim_param!='batch' or out_shape[0].dim_param!='batch':
        raise RuntimeError('ONNX shape/dynamic batch contract differs')
    forbidden={'DFT','FFT','Spectral','Complex','ReduceMin'}
    if any(n.op_type in forbidden or 'fft' in n.op_type.lower() for n in graph.graph.node):
        raise RuntimeError('Training-time spectral logic leaked into inference ONNX')
    cpu=ort.InferenceSession(str(pending),providers=['CPUExecutionProvider'])
    actual=cpu.run(['logits'],{'input':validation_batch.numpy()})[0]
    difference=np.abs(expected-actual)
    np.testing.assert_allclose(expected,actual,rtol=1e-4,atol=1e-5)
    os.replace(pending,path) # publish only after parity passes
    validation={'max_abs_error':float(difference.max()),'mean_abs_error':float(difference.mean()),
        'rtol':1e-4,'atol':1e-5,'provider':'CPUExecutionProvider',
        'input_shape':list(validation_batch.shape),'output_shape':list(actual.shape),
        'opset':17,'val_indices':indices,'onnx_sha256':sha256_file(path)}
    strict_json_write(folder/'onnx_validation.json',validation)
    base={'branch':result['name'],'parameters':sum(p.numel() for p in model.parameters()),
        'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
        'input_resolution':'80x80','pytorch_checkpoint_bytes':(folder/'best.pth').stat().st_size,
        'onnx_model_bytes':path.stat().st_size,'inference_graph':'ordinary MiniFASNetV2 only'}
    efficiency.append({**base,**latency(cpu,'CPUExecutionProvider')})
    if 'CUDAExecutionProvider' in ort.get_available_providers():
        cuda=ort.InferenceSession(str(path),providers=['CUDAExecutionProvider','CPUExecutionProvider'])
        if cuda.get_providers()[0]=='CUDAExecutionProvider':
            efficiency.append({**base,**latency(cuda,'CUDAExecutionProvider')})
    print(result['name'],'ONNX parity',validation,'sizes',base)
pd.DataFrame(efficiency).to_csv(output/'efficiency_metrics.csv',index=False)


In [ ]:
threshold_logit=float(strict_json_load(output/'val_freeze.json')['locked_val_threshold'])
if threshold_logit>=0:
    z=math.exp(-threshold_logit);threshold_probability=1/(1+z)
else:
    z=math.exp(threshold_logit);threshold_probability=z/(1+z)
if not 0<threshold_probability<1:raise RuntimeError('Threshold probability not representable')
model_sha=sha256_file(output/'best.onnx');validation=strict_json_load(output/'onnx_validation.json')
if validation['onnx_sha256']!=model_sha:raise RuntimeError('ONNX validation SHA mismatch')
runtime_contract={'schema':'face_auth_pad_runtime_config_v1','model_file':'best.onnx','model_sha256':model_sha,
 'training_checkpoint_sha256':result['sha'],'model_img_size':80,'input_name':'input','output_name':'logits',
 'input_shape':['batch',3,80,80],'output_shape':['batch',3],'class_names':['real','physical_spoof','digital_spoof'],
 'color_order':'BGR','input_range':'[0,1]','mean':None,'std':None,'apply_gamma':False,
 'bbox_expansion_factor':2.7,'crop_mode':'minifasnet_train_v1','crop_smoothing':False,
 'predictor_threshold_probability':threshold_probability,'calibrated_logit_threshold':threshold_logit,
 'threshold_score_definition':'logit_real - logsumexp(logit_physical, logit_digital)',
 'threshold_source':'CelebA-Spoof Val50K only','onnx_opset':17}
strict_json_write(output/'runtime_config.json',runtime_contract)


In [ ]:
lines=[f'# Full MiniFASNetV2 {BRANCH} experiment','',
 f'Evaluation contract: {EVALUATION_CONTRACT_VERSION}; SHA256 {contract_sha}.','',
 'End-to-end detector-conditioned PAD: SCRFD → 2.7× crop → MiniFASNetV2. GT boxes are never model crops.','',
 'Official PAD-pretrained MiniFASNetV2 feature weights; final project 3-class classifier reinitialized deterministically.','',
 f'Source: CelebA-Spoof full eligible Train {len(train_frame)} after subject-disjoint Val50K reservation and frozen localization hygiene.','',
 'Validation: 50,000 images derived exclusively from official CelebA training metadata. Lowest Val ACER selects checkpoint; highest Val AUC breaks ties. Threshold frozen from Val only.','',
 f'Same-dataset final test: {len(test)} official candidates; {len(test_frame)} SCRFD-scorable; coverage {len(test_frame)/len(test):.6f}.','',
 f'Cross-dataset test: LCC-FASD official evaluation, 7,580 candidates (314 bona fide, 7,266 spoof); {len(lcc_scored)} SCRFD-scorable; coverage {len(lcc_scored)/len(lcc_frame):.6f}.','',
 'A smaller LCC external-development resource was used during early development; its overlap with the official evaluation split is disclosed in the resource bundle.','',
 f'Initial state SHA: {initial_sha}. Best checkpoint SHA: {result["sha"]}. Val threshold: {locked}.','',
 f'AdamW backbone LR {BACKBONE_LR}; head LR {CLASSIFIER_LR}; weight decay {WEIGHT_DECAY}; label smoothing {LABEL_SMOOTHING}; batch {BATCH_SIZE}; epochs {MAX_EPOCHS}; patience {EARLY_STOP_PATIENCE}.','',
 '| Split | Candidates | Scored | Coverage | AUC | APCER | BPCER | ACER/HTER | EER | TPR@1% FPR |','|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for split,m in metrics_all.items():
    lines.append(f'| {split} | {m["candidate_n"]} | {m["n_scored"]} | {m["detector_coverage"]:.6f} | {m["auc"]:.6f} | {m["apcer"]:.6f} | {m["bpcer"]:.6f} | {m["hter" if split=="lcc" else "acer"]:.6f} | {m["eer"]:.6f} | {m["tpr_at_fpr_1pct"]:.6f} |')
lines+=['','PAD metrics are reported over samples successfully localized by the frozen SCRFD detector; detector coverage is reported separately. LCC AUC/EER/TPR are threshold-free target ROC statistics. LCC APCER/BPCER/HTER use the locked CelebA Val50K threshold.','',
 'One seed is descriptive, not a significance test. CPU ONNX latency is an offline proxy and does not establish camera performance.','',
 'See resource_fingerprints.json, config.json, initialization_fingerprint.json, val_freeze.json, predictions, confusion matrices, ONNX parity and runtime_config.json.']
(output/'REPORT.md').write_text('\n'.join(lines)+'\n')
required=['best.pth','last_state.pth','meta.json','config.json','resource_fingerprints.json','initialization_fingerprint.json',
 'training_history.csv','val_predictions.csv','test_predictions.csv','lcc_predictions.csv','val_metrics.json','test_metrics.json','lcc_metrics.json',
 'metrics_summary.csv','metrics_summary.json','val_confusion.json','test_confusion.json','roc_val.png','roc_test.png','roc_lcc.png',
 'best.onnx','onnx_validation.json','runtime_config.json','efficiency_metrics.csv','REPORT.md']
if CSMR_ENABLED:required+=['csmr_diagnostics.csv','class_diagnostics.csv']
missing=[n for n in required if not (output/n).is_file()]
if missing:raise RuntimeError('Missing final artifacts: '+', '.join(missing))
ZIP_PATH=Path(str(output)+'.zip');write_zip(output,ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH) as archive:
    names=set(archive.namelist())
    for n in required:
        if f'{output.name}/{n}' not in names:raise RuntimeError('ZIP missing '+n)
print('ZIP:',ZIP_PATH,'MB:',ZIP_PATH.stat().st_size/1e6);display(FileLink(str(ZIP_PATH)))
